In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
pylab.rcParams.update({'figure.figsize': (18, 8), 'axes.titlesize': 'large'})

import pandas as pd
import numpy as np
import datetime
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../../')

# SFR Kink-Fade Live Screener

BF_6M kink-fading signals using the 0.83 Sharpe production config:
- **Direction:** buy_kink only (belly rate too high, fade it down)
- **Region:** Reds (SFR5-8) primary, full strip for context
- **Entry:** |z| > 2.0 on 60d window
- **Filters:** FOMC 5d blackout, HL gating (3-120d), IMM roll 3d blackout

---
## 1. Run Screener

In [ ]:
from RVUtils.SFRKinkFadeScreener import build_snapshot, KinkFadeScreenerConfig
from RVUtils.SFRKinkFadeScreener._display import (
    render_dashboard_table, render_strip_chart,
    render_zscore_chart, render_filter_panel,
)

config = KinkFadeScreenerConfig(
    entry_zscore=2.0,
    fomc_blackout_days=5,
    roll_blackout_days=3,
    hl_min=3.0,
    hl_max=120.0,
)

snapshot = build_snapshot(config)
print(f'Snapshot as of: {snapshot.as_of}')
print(f'Actionable signals: {snapshot.n_actionable}')

---
## 2. Filter Status

In [ ]:
render_filter_panel(snapshot)

---
## 3. Signal Dashboard

In [ ]:
render_dashboard_table(snapshot)

---
## 4. Strip Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
render_strip_chart(snapshot, ax=ax)
plt.tight_layout()
plt.show()

---
## 5. Z-Score History (Reds)

In [ ]:
from BT.signals.sfr_cal_spread_rv import load_rate_panel, SFRCalSpreadRVConfig, compute_fly_curve
from BT.signals.sfr_kink_fade import compute_zscore_ts
import pytz

NYC = pytz.timezone('America/New_York')
rv_config = SFRCalSpreadRVConfig(
    source=config.source, curve=config.curve,
    n_contracts=config.n_contracts, constant_maturity=True,
    zscore_window=config.zscore_window, vol_window=config.vol_window,
)
start = NYC.localize(datetime.datetime.combine(
    snapshot.as_of - datetime.timedelta(days=400), datetime.time(18, 0)))
try:
    rates = load_rate_panel(rv_config, start=start, end='live')
except Exception:
    end_dt = NYC.localize(datetime.datetime.combine(snapshot.as_of, datetime.time(18, 0)))
    rates = load_rate_panel(rv_config, start=start, end=end_dt)

bf6m = compute_fly_curve(rates, gap=2)
zscore_history = compute_zscore_ts(bf6m, config.zscore_window)

fig, ax = plt.subplots(figsize=(16, 6))
render_zscore_chart(snapshot, zscore_history, ax=ax, n_days=120)
plt.tight_layout()
plt.show()

---
## 6. Detailed View (per-fly filter breakdown)

In [ ]:
print(f'{"Structure":<25s}  {"Z":>6s}  {"Dir":>10s}  {"Reg":>7s}  {"HL":>6s}  z_ok dir  reg  fomc roll hl   ENTRY')
print('-' * 100)
for r in snapshot.results:
    f = r.filters
    hl_str = f'{r.half_life_days:.0f}d' if not np.isnan(r.half_life_days) else 'N/A'
    checks = f'{"Y" if f["z_threshold"] else ".":>3s}  {"Y" if f["direction"] else ".":>3s}  {"Y" if f["region"] else ".":>3s}  {"Y" if f["fomc_blackout"] else "X":>3s}  {"Y" if f["roll_blackout"] else "X":>3s}  {"Y" if f["hl_gating"] else ".":>3s}'
    flag = ' *** ENTRY ***' if r.entry_eligible else ''
    print(f'  {r.structure_id:<23s}  {r.zscore:>+6.2f}  {r.direction:>10s}  {r.region:>7s}  {hl_str:>6s}  {checks}{flag}')